In [ ]:
#%pip install -q google-genai sentence-transformers faiss-cpu PyPDF2

import google.genai as genai
from sentence_transformers import SentenceTransformer
from PyPDF2 import PdfReader
import faiss
import numpy as np

In [ ]:
API_KEY = "GEMINI_API_KEY"

client = genai.Client(api_key=API_KEY)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explique overfitting em português"
)

print(response.text)

In [ ]:
reader = PdfReader("../data/estatistica.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text() + "\n"

In [ ]:
def split_text(text, chunk_size=500):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

documents = split_text(text)

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embedder.encode(documents)

In [ ]:
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(np.array(doc_embeddings))

In [ ]:
def retrieve(query, k=2):
    q_emb = embedder.encode([query])
    distances, indices = index.search(np.array(q_emb), k)
    return [documents[i] for i in indices[0]]

In [ ]:
def ask(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

In [ ]:
def rag(query):
    context = retrieve(query)
    
    prompt = f"""
Use the following context:
{context}

Answer:
{query}
"""
    return ask(prompt)

In [ ]:
query = "Como melhorar a eficiência da cadeia de abastecimento?"

simple = ask(query)
rag_answer = rag(query)

print("SIMPLES:\n", simple)
print("\nRAG:\n", rag_answer)

In [ ]:
print("""
Compare:

- Qual é mais factual?
- Qual usa contexto?
- Qual é mais confiável?
""")